Language: **English (default)** | [Polski](lab.ipynb)

# Lab 14 -- error auditing and decision interpretation (EuroSAT + ResNet18)

Companion material to [`README.md`](README.md). A different problem than
the lecture (there: Fashion-MNIST from scratch). Here: EuroSAT + ResNet18
frozen backbone.

In [ ]:
import collections  # noqa: F401 - used in the TODO below
import time

import matplotlib.pyplot as plt  # noqa: F401 - used in the TODO below
import torch
import torch.nn as nn
import torch.nn.functional as F  # noqa: F401 - used in the TODO below
import torchvision.datasets as tvd
from torchvision.models import ResNet18_Weights, resnet18

torch.manual_seed(0)
torch.set_num_threads(4)
device = torch.device("cpu")
print(f"Using device: {device}")

weights = ResNet18_Weights.IMAGENET1K_V1
preprocess = weights.transforms()
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
BASELINE_GRAY = (0.5 - MEAN) / STD  # mid-gray in normalized space, for occlusion later

full_ds = tvd.EuroSAT(root="data_cache", download=True, transform=preprocess)
CLASS_NAMES = full_ds.classes
print(f"total images: {len(full_ds)}, classes: {CLASS_NAMES}")


def load_subset(indices):
    X = torch.stack([full_ds[i][0] for i in indices])
    y = torch.tensor([full_ds[i][1] for i in indices])
    return X, y


backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
for p in backbone.parameters():
    p.requires_grad = False
N_CLASSES = 10
backbone.fc = nn.Linear(backbone.fc.in_features, N_CLASSES)
backbone.eval()
print(f"trainable params: {sum(p.numel() for p in backbone.parameters() if p.requires_grad)}")


def extract_features(model, X, batch_size=64):
    feats = []
    orig_fc = model.fc
    model.fc = nn.Identity()
    with torch.no_grad():
        for i in range(0, X.shape[0], batch_size):
            feats.append(model(X[i:i + batch_size]))
    model.fc = orig_fc
    return torch.cat(feats, dim=0)

## 1. Data and the train/validation/test split -- your task

In [ ]:
# TODO:
# 1. torch.manual_seed(0); perm = torch.randperm(len(full_ds))
# 2. train_idx, val_idx, test_idx = perm[:1800], perm[1800:2200], perm[2200:2600]
#    (or another disjoint split, justify the sizes)
# 3. VERIFY PROGRAMMATICALLY no overlap (e.g. via set intersection)
# 4. print the class distribution in each subset (collections.Counter)
# 5. comment: what would happen if you used the same split to choose
#    hyperparameters AND for the final evaluation?
raise NotImplementedError("TODO: leak-free data split")

## 2. Training the head (provided + a short task)

In [ ]:
def train_head(feat_train, y_train, n_classes=10, epochs=30, batch_size=64, lr=1e-3, seed=0):
    torch.manual_seed(seed)
    head = nn.Linear(feat_train.shape[1], n_classes)
    opt = torch.optim.Adam(head.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    n = feat_train.shape[0]
    t0 = time.time()
    for epoch in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            opt.zero_grad()
            out = head(feat_train[idx])
            loss = loss_fn(out, y_train[idx])
            loss.backward()
            opt.step()
    train_time = time.time() - t0
    return head, train_time

In [ ]:
# TODO:
# X_train, y_train = load_subset(train_idx.tolist())  # noqa: F821
# X_val, y_val = load_subset(val_idx.tolist())  # noqa: F821
# X_test, y_test = load_subset(test_idx.tolist())  # noqa: F821
# feat_train = extract_features(backbone, X_train)
# feat_val = extract_features(backbone, X_val)
# feat_test = extract_features(backbone, X_test)
# head, train_time = train_head(feat_train, y_train)  # TRAIN ONLY on train_idx!
# backbone.fc = head
# print(f"train_time: {train_time:.2f}s")
raise NotImplementedError("TODO: run head training on train_idx")

## 3. Evaluation on an independent set -- your task

In [ ]:
# TODO:
# with torch.no_grad():
#     test_logits = backbone.fc(feat_test)  # head already trained, reuse precomputed features
#     test_pred = test_logits.argmax(1)
#     test_probs = F.softmax(test_logits, dim=1)
# test_acc = (test_pred == y_test).float().mean().item()
#
# C = 10
# confusion = torch.zeros(C, C, dtype=torch.long)
# for t, p in zip(y_test.tolist(), test_pred.tolist()): confusion[t, p] += 1
#
# per-class precision/recall/F1, macro-F1 -- reuse the pattern from the lecture
# plt.imshow(confusion.numpy(), cmap="Blues") + labels + colorbar
raise NotImplementedError("TODO: evaluation and confusion matrix on test_idx")

## 4. The most frequent and most costly errors -- your task

In [ ]:
# TODO: from the confusion matrix (excluding the diagonal), find the TWO most
# frequently confused class pairs. Comment: does one of these pairs have a
# different "cost" of confusion than another?
raise NotImplementedError("TODO: most frequent errors")

## 5. Confidence of correct and wrong predictions -- your task

In [ ]:
# TODO:
# confidences = test_probs.max(dim=1).values
# correct_mask = test_pred == y_test
# histogram confidences[correct_mask] vs confidences[~correct_mask]
# find: idx of the most confident WRONG prediction (confidences[~correct_mask].argmax(),
# remember to convert the index back to the full test_idx)
raise NotImplementedError("TODO: confidence analysis")

## 6. Choosing samples for detailed analysis -- your task

In [ ]:
# TODO: define an EXPLICIT procedure (record EXACTLY how each example was
# chosen, in a comment):
# a) correct prediction with HIGH confidence
# b) error with LOW confidence
# c) error with HIGH confidence (misleading)
raise NotImplementedError("TODO: choosing samples for analysis")

## 7. Grad-CAM -- your task

The backbone is frozen -- `model.eval()` is not enough, you need
`x.requires_grad_(True)` on the input (see the lecture).

In [ ]:
# TODO:
# class GradCAM:
#     def __init__(self, model, target_layer):
#         ...  # forward_hook + register_full_backward_hook
#     def __call__(self, x, target_class):
#         ...  # alpha = grad.mean(dim=(2,3)); weighted sum; relu; interpolate; normalize
#              # (handle zero-spread map: check (max-min) > eps before dividing)
#     def remove_hooks(self):
#         ...
raise NotImplementedError("TODO: Grad-CAM implementation")

## 8. Comparing heatmaps and occlusion sensitivity -- your task

In [ ]:
# TODO:
# 1. for each of the 3 examples (section 6): compute Grad-CAM on backbone.layer4
#    for the PREDICTED class, on x.clone().requires_grad_(True)
# 2. draw image + heatmap + overlay for each
# 3. for example (a) [correct, high confidence]: compute occlusion sensitivity
#    (patch=32, stride=32, baseline=BASELINE_GRAY -- NOT zero)
# 4. comment: do the Grad-CAM and occlusion regions agree or differ?
raise NotImplementedError("TODO: Grad-CAM + occlusion on the chosen samples")

## 9. Subgroup analysis OR robustness -- your task (choose ONE)

See `README.md` section 9 for the full instructions for both options.

In [ ]:
# TODO: choose OPTION A (brightness) or OPTION B (horizontal flip), implement it,
# and answer the critical question from the README (confounding by class /
# generalization to a real distribution shift)
raise NotImplementedError("TODO: subgroup or robustness analysis")

## 10. Diagnostic task

The following code (provided, run it) tries to compute Grad-CAM, but has
two related problems.

In [ ]:
class GradCAMColleague:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        self.fwd_handle = target_layer.register_forward_hook(self._save_activation)
        self.bwd_handle = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def __call__(self, x, target_class):
        # BUG 1: wrapped in no_grad "for speed" by a well-meaning colleague
        with torch.no_grad():
            self.model.zero_grad()
            logits = self.model(x)
            score = logits[:, target_class].sum()
            score.backward()
        return self.gradients

    def remove_hooks(self):
        self.fwd_handle.remove()
        self.bwd_handle.remove()


gradcam_colleague = GradCAMColleague(backbone, backbone.layer4)
try:
    result = gradcam_colleague(X_test[0:1], target_class=0)  # noqa: F821
    print("no error (unexpected)")
except RuntimeError as e:
    print(f"RuntimeError: {e}")

In [ ]:
# TODO:
# 1. comment: exactly which exception, and why (refer to the lecture)?
# 2. fix BUG 1: remove torch.no_grad() from __call__ -- run again, still an error?
# 3. diagnose BUG 2 (frozen backbone + no requires_grad on the input)
# 4. fix BOTH, verify the gradient is finite and nonzero for at least one of
#    your examples from section 6
raise NotImplementedError("TODO: diagnosing and fixing both problems")

## 11. Final report -- your task

**1. How well does the model classify each class? (specific numbers)**

_(Your answer here.)_

**2. Which errors are most frequent? (specific pairs, from the confusion matrix)**

_(Your answer here.)_

**3. Does the model sometimes show high confidence in wrong decisions? (a specific example)**

_(Your answer here.)_

**4. Which input regions does Grad-CAM point to? (for your 3 examples)**

_(Your answer here.)_

**5. Does the perturbation-based analysis (occlusion) support this interpretation?**

_(Your answer here.)_

**6. Does the model's behavior change after your chosen input modification (section 9)?**

_(Your answer here.)_

**7. What should be checked before using this model in practice?**

_(Your answer here.)_